In [2]:
import numpy as np
import tritonclient.http as httpclient
from tritonclient.utils import np_to_triton_dtype

In [7]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import tritonclient.http as httpclient
from tritonclient.utils import np_to_triton_dtype


@dataclass
class TritonModelClient:
    """
    Small wrapper around Triton's HTTP client.

    Example:
        client = TritonModelClient(
            url="triton-server:8000",
            model_name="recipe_model",
            model_version="1",
        )

        result = client.infer(
            inputs={
                "input_ids": np.array([[1, 2, 3]], dtype=np.int32),
                "attention_mask": np.array([[1, 1, 1]], dtype=np.int32),
            },
            output_names=["logits"],
        )
    """
    url: str
    model_name: str
    model_version: str | None = None
    verbose: bool = False

    def __post_init__(self) -> None:
        self._client = httpclient.InferenceServerClient(
            url=self.url,
            verbose=self.verbose,
        )

    def is_server_live(self) -> bool:
        return self._client.is_server_live()

    def is_server_ready(self) -> bool:
        return self._client.is_server_ready()

    def is_model_ready(self) -> bool:
        return self._client.is_model_ready(
            model_name=self.model_name,
            model_version=self.model_version,
        )

    def infer(
        self,
        inputs: dict[str, np.ndarray],
        output_names: list[str] | None = None,
        request_id: str | int | None = None,
    ) -> dict[str, np.ndarray]:
        if not inputs:
            raise ValueError("inputs cannot be empty")
    
        triton_inputs: list[httpclient.InferInput] = []
        for input_name, array in inputs.items():
            if not isinstance(array, np.ndarray):
                raise TypeError(
                    f"Input '{input_name}' must be a numpy array, got {type(array)}"
                )
    
            infer_input = httpclient.InferInput(
                input_name,
                array.shape,
                np_to_triton_dtype(array.dtype),
            )
            infer_input.set_data_from_numpy(array)
            triton_inputs.append(infer_input)
    
        triton_outputs = None
        if output_names is not None:
            triton_outputs = [httpclient.InferRequestedOutput(name) for name in output_names]
    
        safe_request_id = None if request_id is None else str(request_id)
    
        response = self._client.infer(
            model_name=self.model_name,
            model_version=self.model_version,
            inputs=triton_inputs,
            outputs=triton_outputs,
            request_id=safe_request_id,
        )
    
        if output_names is None:
            output_names = [output["name"] for output in response.get_response().get("outputs", [])]
    
        return {name: response.as_numpy(name) for name in output_names}

In [18]:
import numpy as np
import tritonclient.http as httpclient


client = httpclient.InferenceServerClient(url="triton:8000")

text_tensor = np.array([["fix this: Title: Valentine's Thins\nIngredients: 36 RITZ Crackers with Whole Wheat | 2 pkg. (4 oz. each) BAKER'S Semi-Sweet Chocolate, broken into pieces, melted | 36 berry-shaped gummy candies (about 2/3 cup or 4 oz.) | 2 oz. BAKER'S White Chocolate, chopped\nInstructions: Dip crackers in melted semi-sweet chocolate, turning to completely coat each cracker. | Carefully scrape off excess chocolate. | Place in single layer on waxed paper-covered baking sheets. | Immediately top with candies. | Microwave white chocolate in microwaveable bowl on HIGH 1 min. | or until melted, stirring after 30 sec. | Drizzle over crackers. | Refrigerate 15 min. | or until firm."]], dtype=object)

infer_input = httpclient.InferInput(
    "INPUT_TEXT",
    text_tensor.shape,
    "BYTES",
)
infer_input.set_data_from_numpy(text_tensor)

requested_output = httpclient.InferRequestedOutput("OUTPUT_TEXT")

response = client.infer(
    model_name="recipe_model",
    inputs=[infer_input],
    outputs=[requested_output],
    # request_id="1",   # or omit this line entirely
)

output = response.as_numpy("OUTPUT_TEXT")
print(output)

[[b": Title: Valentine's Thins Ingredients: 36 RITZ Crackers with Whole Wheat | 2 pkg. (4 oz. each) BAKER'S Semi-Sweet Chocolate, broken into pieces, melted | 36 berry-shaped gummy"]]
